# Monthly_mrr

Revenue we reasonably expect next month if nothing changes. Given everything we know about customer behaviour and revenue movement this month, how much recurring revenue will the company reliably generate next month?

Features: current_mrr, expansion_mrr, contraction_mrr, churned_mrr, reactivation_mrr, active_subscriptions, trial_to_paid_conversions, renewal_count, churn_count, upgrade_count, downgrade_count, average_seats, enterprise_subscription_count, support_ticket_count.

Label: total_mrr_next_month

So, it is just predict label based on features.


In [0]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number, to_timestamp, lit, create_map, upper, trim
from pyspark.sql.types import DecimalType
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from functools import reduce
from itertools import chain
import sklearn
print(pyspark.__version__)

In [0]:
builder = (
    SparkSession.builder
    .appName("user_bronze")
)
spark = builder.getOrCreate()

In [0]:
silver_db = "/Volumes/datalake_catalog/datalake_schema/silver/"
gold_db = "datalake_catalog.gold"
output_table = f"{gold_db}.mrr"
snapshot_date = F.current_date()
cutoff_date = F.lit("2026-04-30").cast("date")

In [0]:
users = (
    spark.read
    .format("delta")
    .load(silver_db+"/users")
)
products = (
    spark.read
    .format("delta")
    .load(silver_db+"/products")
)
plans = (
    spark.read
    .format("delta")
    .load(silver_db+"/plans")
)
subs = (
    spark.read
    .format("delta")
    .load(silver_db+"/subscriptions")
)

changes = (
    spark.read
    .format("delta")
    .load(silver_db+"/subscription_changes")
)
payments = (
    spark.read
    .format("delta")
    .load(silver_db+"/payments")
)
licenses = (
    spark.read
    .format("delta")
    .load(silver_db+"/licenses")
)
allocations = (
    spark.read
    .format("delta")
    .load(silver_db+"/license_allocations")
)
usage = (
    spark.read
    .format("delta")
    .load(silver_db+"/usage_events")
)
tickets = (
    spark.read
    .format("delta")
    .load(silver_db+"/support_tickets")
)

## Theory + Business domain - currently: Company-level MRR


We use company-level MRR as the main business metric, and customer/account-level MRR for deeper analysis. User-level MRR is less common unless each user is the actual paying customer.

Recurring revenue is: money the company expects to receive again and again on a regular basis because the customer stays subscribed.
For each month t, build one company-level row using only data known up to the end of month t.

Monthly rule:
<br>
Use month-end snapshot + monthly event aggregation:
+ Snapshot features = state at the end of month: current_mr, active_subscriptions, average_seats, enterprise_subscription_count.
+ Flow/movement features = events during the month: expansion_mrr, contraction_mrr, churned_mrr, reactivation_mrr, trial_to_paid_conversions, renewal_count, churn_count, upgrade_count, downgrade_count, support_ticket_count

Methods for inferring each feature:
+ month: This is the reporting month for the company-level MRR metrics. Each row represents one calendar month, and all MRR movement values in that row are calculated for that month. In the code, this month comes from the payment coverage months after expanding successful payments and then aggregating all user-level MRR movements into one company-level monthly view.

+ current_mrr: This is the total monthly recurring revenue active in that month across the whole company. At user level, each user’s current_mrr is the sum of monthly-equivalent recurring revenue from all of that user’s subscriptions that are both covered by a successful payment and active at that month end. Then at company level, current_mrr is simply the sum of all users’ current_mrr in that month. In short, this tells stakeholders: “How much recurring revenue is currently live this month?”

+ new_mrr: This is the recurring revenue in the current month coming from users who had no MRR in the previous month but have positive MRR in the current month. In other words, these are users who newly started contributing recurring revenue this month. At user level, if prev_mrr = 0 and current_mrr > 0, then the full current_mrr for that user in that month is classified as new_mrr. At company level, new_mrr is the sum of all such user-level values for the month. This answers: “How much new recurring revenue did we gain this month from newly paying users?”

+ expansion_mrr: This is the increase in recurring revenue from existing users compared with the previous month. A user contributes to expansion_mrr only if the user already had positive MRR last month and the user’s current month MRR is now higher. The amount counted is only the positive increase: current_mrr - prev_mrr. At company level, expansion_mrr is the total of these positive increases across all existing users for the month. This tells the business: “How much extra recurring revenue did existing users add this month through upgrades, more plans, or higher-value subscriptions?”

+ contraction_mrr: This is the decrease in recurring revenue from existing users who are still active, but whose MRR is lower than in the previous month. A user contributes to contraction_mrr if prev_mrr > 0, current_mrr > 0, and current_mrr < prev_mrr. The amount counted is the drop: prev_mrr - current_mrr. At company level, contraction_mrr is the sum of all such decreases across users for that month. This answers: “How much recurring revenue did we lose this month from existing users downgrading or reducing value, without fully churning?”

+ churned_mrr: This is the recurring revenue lost from users who had positive MRR in the previous month but have zero MRR in the current month. In other words, these are users who were contributing recurring revenue before and are now fully gone from the recurring revenue base. At user level, if prev_mrr > 0 and current_mrr = 0, then the full previous month value is counted as churned_mrr. At company level, churned_mrr is the total of those lost values for the month. This tells stakeholders: “How much recurring revenue completely dropped out this month due to full churn?”
+ The relationship: Current MRR this month = Current MRR last month + New MRR + Expansion MRR - Contraction MRR - Churned MRR

In [0]:
# =========================================================
# Join subscriptions with plans
# =========================================================
subs_plan = (
    subs
    .select(
        "user_id",
        "subscription_id",
        "plan_id",
        "start_date",
        "end_date",
        "status",
        "created_at"
    )
    .join(
        plans.select("plan_id", "billing_cycle", "price", "currency"),
        on="plan_id",
        how="left"
    )
    .withColumn("billing_cycle", F.lower(F.trim(F.col("billing_cycle"))))
)

# =========================================================
# Keep only successful payments
#    Use ONE consistent amount field for MRR reporting
#    Here I use `amount` because it appears to be normalized/base currency
# =========================================================
payments_success = (
    payments
    .filter(F.lower(F.trim(F.col("payment_status"))) == "success")
    .filter(F.col("payment_date").isNotNull())
    .dropDuplicates(["payment_id"])
    .select(
        "subscription_id",
        "payment_id",
        "payment_date",
        "amount"   # <-- replace if another normalized reporting amount is preferred
    )
)

# =========================================================
# Join subs + plans + successful payments
# =========================================================
subs_plan_pay = (
    subs_plan
    .join(payments_success, on="subscription_id", how="left")
)

# =========================================================
# Derive payment coverage and monthly-equivalent MRR
# =========================================================
paid_coverage = (
    subs_plan_pay
    .filter(F.col("payment_id").isNotNull())
    .filter(F.col("billing_cycle").isin("monthly", "annual"))
    .withColumn(
        "months_covered",
        F.when(F.col("billing_cycle") == "monthly", F.lit(1))
         .when(F.col("billing_cycle") == "annual", F.lit(12))
    )
    .withColumn(
        "monthly_equiv_mrr",
        F.when(F.col("billing_cycle") == "monthly", F.col("amount"))
         .when(F.col("billing_cycle") == "annual", F.col("amount") / F.lit(12.0))
    )
    .withColumn("payment_month", F.trunc(F.col("payment_date"), "month"))
    .withColumn("coverage_end_month", F.add_months(F.col("payment_month"), F.col("months_covered") - 1))
)

# =========================================================
# Expand each payment into all months it covers
# =========================================================
expanded_paid_months = (
    paid_coverage
    .withColumn(
        "month",
        F.explode(
            F.expr("sequence(payment_month, coverage_end_month, interval 1 month)")
        )
    )
    .withColumn("month_end", F.last_day(F.col("month")))
)

# =========================================================
# Keep only subscription-months where subscription is active at month end
#    Use >= if end_date means the last active day is included
# =========================================================
active_paid_months = (
    expanded_paid_months
    .filter(F.col("start_date").isNotNull())
    .filter(F.col("start_date") <= F.col("month_end"))
    .filter(
        F.col("end_date").isNull() |
        (F.col("end_date") >= F.col("month_end"))
    )
)

# =========================================================
# Deduplicate at subscription_id + month
#    Use MAX to avoid double-counting overlapping duplicate payments
# =========================================================
subscription_month_mrr = (
    active_paid_months
    .groupBy("user_id", "subscription_id", "month")
    .agg(
        F.max("monthly_equiv_mrr").alias("subscription_month_mrr")
    )
)

# =========================================================
# Aggregate to user-level current MRR per month
# =========================================================
user_month_mrr = (
    subscription_month_mrr
    .groupBy("user_id", "month")
    .agg(
        F.sum("subscription_month_mrr").alias("current_mrr")
    )
)

# =========================================================
# Build a full user-month spine so missing months become 0 MRR
#    This is important for correct expansion / churn logic
# =========================================================
min_max_month = user_month_mrr.agg(
    F.min("month").alias("min_month"),
    F.max("month").alias("max_month")
).collect()[0]

min_month = min_max_month["min_month"]
max_month = min_max_month["max_month"]

calendar_months = (
    spark.sql(
        f"""
        SELECT explode(
            sequence(
                to_date('{min_month}'),
                to_date('{max_month}'),
                interval 1 month
            )
        ) AS month
        """
    )
)

all_users = subs.select("user_id").distinct()

user_month_spine = all_users.crossJoin(calendar_months)

user_month_mrr_full = (
    user_month_spine
    .join(user_month_mrr, on=["user_id", "month"], how="left")
    .fillna({"current_mrr": 0.0})
)

# =========================================================
# Previous month MRR per user
# =========================================================
w = Window.partitionBy("user_id").orderBy("month")

user_mrr_movement = (
    user_month_mrr_full
    .withColumn("prev_mrr", F.lag("current_mrr").over(w))
    .withColumn("prev_mrr", F.coalesce(F.col("prev_mrr"), F.lit(0.0)))
)

# =========================================================
# Derive MRR movements
# =========================================================
user_mrr_movement = (
    user_mrr_movement
    .withColumn(
        "expansion_mrr",
        F.when(
            (F.col("prev_mrr") > 0) &
            (F.col("current_mrr") > F.col("prev_mrr")),
            F.col("current_mrr") - F.col("prev_mrr")
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "new_mrr",
        F.when(
            (F.col("prev_mrr") == 0) &
            (F.col("current_mrr") > 0),
            F.col("current_mrr")
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "contraction_mrr",
        F.when(
            (F.col("prev_mrr") > 0) &
            (F.col("current_mrr") > 0) &
            (F.col("current_mrr") < F.col("prev_mrr")),
            F.col("prev_mrr") - F.col("current_mrr")
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "churned_mrr",
        F.when(
            (F.col("prev_mrr") > 0) &
            (F.col("current_mrr") == 0),
            F.col("prev_mrr")
        ).otherwise(F.lit(0.0))
    )
)

# =========================================================
# Company-level monthly totals
# =========================================================
df1 = (
    user_mrr_movement
    .groupBy("month")
    .agg(
        F.sum("current_mrr").alias("current_mrr"),
        F.sum("new_mrr").alias("new_mrr"),
        F.sum("expansion_mrr").alias("expansion_mrr"),
        F.sum("contraction_mrr").alias("contraction_mrr"),
        F.sum("churned_mrr").alias("churned_mrr")
    )
    .orderBy("month")
)

# =========================================================
# Final outputs
# =========================================================
# 1) user_month_mrr_full       -> user-level current MRR by month
# 2) user_mrr_movement         -> user-level MRR movement including expansion
# 3) mrr_movement_by_month     -> company-level monthly totals

## active_subscriptions

In [0]:

active_subscriptions_by_month = (
    subscription_month_mrr
    .groupBy("month")
    .agg(
        F.countDistinct("subscription_id").alias("active_subscriptions")
    )
    .orderBy("month")
)

df2 = (
    df1
    .join(active_subscriptions_by_month, on="month", how="left")
    .fillna({"active_subscriptions": 0})
    .orderBy("month")
)


## trial_to_paid_conversions,

subs + plans
→ know product_id and whether the subscription is trial or paid

payments
→ know which subscription has successful payment

Then:
paid subscription is convert_to_paid = 1
if the same user had a trial for the same product before that paid payment

In [0]:
# =========================================================
# Join subscriptions with plans
# This gives each subscription product_id, tier, price
# =========================================================
subs_plans = (
    subs.drop("price").alias("s")
    .join(
        plans.select(
            F.col("plan_id").alias("plan_join_id"),
            "product_id",
            "tier",
            "price"
        ).alias("p"),
        F.col("s.plan_id") == F.col("p.plan_join_id"),
        "left"
    )
    .drop("plan_join_id")
)

# =========================================================
# Create is_trial column
# =========================================================
subs_plans = (
    subs_plans
    .withColumn("tier_lc", F.lower(F.trim(F.col("tier"))))
    .withColumn(
        "is_trial",
        F.when(
            F.col("tier_lc").isin("trial", "free trial", "free_trial"),
            F.lit(1)
        ).when(
            F.col("price") == 0,
            F.lit(1)
        ).otherwise(F.lit(0))
    )
)

# =========================================================
# Get first successful payment for each subscription
# =========================================================
successful_payments = (
    payments
    .filter(F.col("payment_status") == "success")
    .filter(F.col("payment_date").isNotNull())
    .groupBy("subscription_id")
    .agg(
        F.min("payment_date").alias("first_success_payment_date")
    )
)

# =========================================================
# Join successful payment info back to subscriptions
# =========================================================
subs_full = (
    subs_plans
    .join(successful_payments, on="subscription_id", how="left")
)

# =========================================================
# Separate trial subscriptions
# =========================================================
trial_subs = (
    subs_full
    .filter(F.col("is_trial") == 1)
    .select(
        F.col("user_id").alias("trial_user_id"),
        F.col("product_id").alias("trial_product_id"),
        F.col("start_date").alias("trial_start_date"),
        F.col("end_date").alias("trial_end_date")
    )
)

# =========================================================
# Separate paid subscriptions with successful payment
# =========================================================
paid_subs = (
    subs_full
    .filter(F.col("is_trial") == 0)
    .filter(F.col("first_success_payment_date").isNotNull())
    .select(
        "subscription_id",
        "user_id",
        "product_id",
        "start_date",
        "first_success_payment_date"
    )
)

# =========================================================
# Check whether paid subscription had previous trial
# Same user + same product + payment after trial
# =========================================================
converted_paid_subs = (
    paid_subs.alias("paid")
    .join(
        trial_subs.alias("trial"),
        (
            (F.col("paid.user_id") == F.col("trial.trial_user_id")) &
            (F.col("paid.product_id") == F.col("trial.trial_product_id")) &
            (
                F.col("paid.first_success_payment_date") >= 
                F.coalesce(F.col("trial.trial_end_date"), F.col("trial.trial_start_date"))
            )
        ),
        "inner"
    )
    .select("paid.subscription_id")
    .distinct()
    .withColumn("convert_to_paid", F.lit(1))
)

# =========================================================
# Join result back to original subscription table
# =========================================================
subs_with_conversion = (
    subs_full
    .drop("convert_to_paid")
    .join(converted_paid_subs, on="subscription_id", how="left")
    .withColumn("convert_to_paid", F.coalesce(F.col("convert_to_paid"), F.lit(0)))
    .drop("tier_lc")
)

In [0]:
df3_temp = (
    subs_with_conversion
    .filter(F.col("convert_to_paid") == 1)
    .withColumn("month", F.date_trunc("month", F.col("first_success_payment_date")))
    .groupBy("month")
    .agg(
        F.countDistinct("subscription_id").alias("num_conversions")
    )
    .orderBy("month")
)

df3 = df2.join(df3_temp, on="month", how="left").fillna(0)

A renewal is a successful payment for an existing paid subscription after the initial paid payment. first successful payment = initial payment
second, third, fourth successful payments = renewals.

We count the "reactivate" as renewal too, and successful payment after a trial is not renewal.

=> we group by user_id + product_id (because each product has many tiers), then, we will count, if there is 2 successful payment, then we count it as renewal.

In [0]:
df4_temp = payments.withColumn("month", F.date_trunc("month", F.col("payment_date")))\
  .filter(F.col("payment_status")=="success")\
  .select("subscription_id", "payment_status",
  "month").alias("p").join(
  subs.alias("s"),
  on="subscription_id",
  how="inner"
).join(
  plans.select("plan_id","product_id"),
  on="plan_id",
  how="inner"
).groupBy("user_id", "product_id").agg(
  F.count("payment_status").alias("num_payments"),
  F.collect_list("month").alias("month")
)

In [0]:
df4 = (
    df4_temp
    # sort the month list first, because collect_list does not guarantee order
    .withColumn("month_sorted", F.sort_array(F.col("month")))
    
    # remove the first element
    .withColumn(
        "renewal_months",
        F.slice(
            F.col("month_sorted"),
            2,
            F.size(F.col("month_sorted")) - 1
        )
    )
    
    # turn remaining months into rows
    .withColumn("month", F.explode(F.col("renewal_months")))
    
    # keep useful columns
    .select(
        "user_id",
        "product_id",
        "num_payments",
        "month"
    )
).filter(F.col("num_payments") > 1)\
.groupBy("month")\
.agg(F.countDistinct("user_id").alias("num_renewals"))\
.join(df3, on="month", how="right")\
.fillna(0)

## Churn count

We just need to use a cutoff to give a row a label churn or not, and then, we just group by month.

In [0]:
df5_temp = (
    subs
    .withColumn(
        "churn",
        F.when(
            (
                F.lower(F.col("status")).isin("cancelled", "canceled", "expired")
            )
            |
            (
                F.col("end_date").isNotNull()
                # &
                # (F.to_date(F.col("end_date")) <= cutoff_date)
            ),
            F.lit(1)
        ).otherwise(F.lit(0))
    )
    .withColumn("month", F.date_trunc("month", F.col("end_date")))
    .filter(F.col("month").isNotNull())
    .groupBy("month")
    .agg(
        F.sum("churn").alias("num_churn")
    )
    .orderBy("month")
)

In [0]:
df5 = df5_temp.join(
    df4,
    on="month",
    how="left"
).fillna(0)

In [0]:
df6_temp = (
    changes
    .withColumn("month", F.date_trunc("month", F.col("change_date")))
    .groupBy("month")
    .agg(
        F.sum(F.when(F.lower(F.col("change_type")) == "upgrade", 1).otherwise(0)).alias("num_upgrades"),
        F.sum(F.when(F.lower(F.col("change_type")) == "downgrade", 1).otherwise(0)).alias("num_downgrades"),
        F.sum(F.when(F.lower(F.col("change_type"))=="initial", 1).otherwise(0)).alias("initial")
    )
    .orderBy("month")
)

In [0]:
df6 = df5.join(
    df6_temp,
    on="month",
    how="left"
).fillna(0)

### enterprise_subscription_count

Count all enterprise subscriptions in that month, no need to care whether or not trial (at this moment).

Count for successful payments only.

In [0]:
df7_temp = subs.select("user_id", "subscription_id")\
  .join(payments.filter(F.col("payment_status")=="success").select("subscription_id", "payment_date"),
        on="subscription_id",how="inner")\
  .join(
    users.select("user_id", "is_enterprise"),
    on="user_id",
    how="left"
).withColumn("month", F.date_trunc("month", F.col("payment_date")))\
  .groupBy("month").agg(
    F.sum(F.when(F.col("is_enterprise")==True, 1).otherwise(0)).alias("num_enterprise")
  )

In [0]:
df7 = df6.join(
    df7_temp,
    on="month",
    how="left"
).fillna(0)

## support_ticket_count

Because there are month which are not in the df7, the total number of tickets might be less than total of tickets after join, this happens mostly because there are month in the tickets that do not exist in the df7 (error somewhere)

In [0]:
df8 = tickets.withColumn("month", F.date_trunc("month", F.col("created_at"))).\
    groupBy("month").agg(F.count("ticket_id").alias("num_tickets")).\
    join(df7, on="month", how="right").fillna(0)

In [0]:
display(df8)

In [0]:
(
    df8
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(output_table)
)

print(f"Saved to {output_table}")